In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q13/q13_rewrite/checkpoints/pre_cell_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###

# 1) Count orders per customer (exclude special requests)
cust_order_counts = (
    orders
    .loc[
        ~orders["O_COMMENT"].str.contains(r"special[\S|\s]*requests"),
        "O_CUSTKEY"
    ]
    .value_counts()
)

# 2) Build distribution of counts for customers with ≥1 order
dist = (
    cust_order_counts
    .value_counts()
    .rename("CUSTDIST")
    .reset_index()
)
dist.columns = ["C_COUNT", "CUSTDIST"]

# compute how many customers had zero orders
zeros = len(customer) - len(cust_order_counts)
if zeros:
    # use concat instead of append (unsupported in cudf.DataFrame)
    zero_df = pd.DataFrame({"C_COUNT": [0], "CUSTDIST": [zeros]})
    dist = pd.concat([dist, zero_df], ignore_index=True)

# 3) Sort and assign to total
total = dist.sort_values(["CUSTDIST", "C_COUNT"], ascending=[False, False])